# 01 — Coleta SciELO por lista de keywords

Este notebook coleta resultados da busca SciELO para uma lista de termos.

## O que ele faz

1. Lê uma lista de `KEYWORDS`.
2. Busca cada termo na coleção Brasil da SciELO (`filter[in][]=scl`).
3. Extrai título, link, autores, periódico, ano, DOI e resumo da página de resultados.
4. Salva um CSV/JSON por termo em uma pasta padrão.
5. Salva também uma base combinada.

## Estrutura de saída

```text
project_dir/
├── data/
│   └── 01_search/
│       ├── scielo_elites_search_results.csv
│       ├── scielo_classes_dominantes_search_results.csv
│       └── scielo_search_results_combined.csv
└── logs/
    └── debug_html/
```

In [ ]:
# Instalação, caso precise:
# !pip install selenium webdriver-manager beautifulsoup4 pandas tqdm lxml unidecode

In [ ]:
import json
import math
import re
import time
from pathlib import Path
from urllib.parse import urlencode

import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
from unidecode import unidecode

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

## 1. Configuração geral

Ajuste apenas esta célula para rodar o projeto em outro diretório ou com outra lista de termos.

In [ ]:
# Use Path.cwd() se o notebook estiver na pasta do projeto.
PROJECT_DIR = Path.cwd()

# Se quiser forçar manualmente:
# PROJECT_DIR = Path('/Users/hdnovak/ic_usp_2025_socio/IC-H2_2025-USP-Extracao-de-artigos')

SEARCH_DIR = PROJECT_DIR / 'data' / '01_search'
LOG_DIR = PROJECT_DIR / 'logs' / 'debug_html'

SEARCH_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

KEYWORDS = [
    'elites',
    'classes dominantes',
    'classes dirigentes',
    'elites dirigentes',
    'elites financeiras',
    'sociologia das elites',
]

COLLECTION = 'scl'       # scl = SciELO Brasil
LANGUAGE = 'pt'
PAGE_SIZE = 15           # a busca SciELO no HTML validado usa 15 resultados por página
MAX_PAGES = None         # None = tenta calcular pelo total de resultados; use inteiro para limitar
HEADLESS = False         # deixe False para debug; depois pode mudar para True
SLEEP_SECONDS = 3

## 2. Funções auxiliares

In [ ]:
def clean_text(value):
    """Normaliza espaços e trata valores vazios."""
    if value is None:
        return ''

    return re.sub(r'\s+', ' ', str(value)).strip()


def slugify_keyword(keyword):
    """Cria um nome seguro para arquivos a partir de uma keyword."""
    slug = unidecode(str(keyword).lower())
    slug = re.sub(r'[^a-z0-9]+', '_', slug)
    slug = re.sub(r'_+', '_', slug).strip('_')
    return slug or 'keyword'


def extract_year_from_source(source_text):
    """Extrai ano de textos do bloco de fonte/periódico."""
    match = re.search(r'\b(19\d{2}|20\d{2})\b', clean_text(source_text))
    return match.group(1) if match else ''


def make_driver(headless=False):
    """Cria um Chrome controlado por Selenium."""
    options = Options()

    if headless:
        options.add_argument('--headless=new')

    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1400,1000')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_experimental_option('excludeSwitches', ['enable-automation'])
    options.add_experimental_option('useAutomationExtension', False)

    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options,
    )


def build_scielo_search_url(keyword, page_index=0, page_size=15, collection='scl', language='pt'):
    """Monta URL de busca da SciELO."""
    params = {
        'q': keyword,
        'lang': language,
        'filter[in][]': collection,
        'from': page_index * page_size,
        'count': page_size,
        'output': 'site',
        'sort': '',
    }

    return 'https://search.scielo.org/?' + urlencode(params, doseq=True)

## 3. Parser da página de resultados

O HTML da SciELO validado usa:

- resultados em `#ResultArea .results .item`
- título em `strong.title`
- autores em `.line.authors a.author`
- periódico/ano em `.line.source`
- DOI em `.DOIResults`
- resumo em `.abstract`

In [ ]:
def parse_scielo_result_item(item):
    """Extrai um resultado individual da busca SciELO."""
    title_tag = item.select_one('strong.title')
    title = clean_text(title_tag.get_text(' ', strip=True)) if title_tag else ''

    link = ''
    if title_tag:
        link_tag = title_tag.find_parent('a')
        if link_tag:
            link = link_tag.get('href', '') or ''

    authors = [
        clean_text(author.get_text(' ', strip=True))
        for author in item.select('.line.authors a.author')
    ]

    source_tag = item.select_one('.line.source')
    source_text = clean_text(source_tag.get_text(' ', strip=True)) if source_tag else ''

    journal_tag = item.select_one('.line.source a.openJournalInfo')
    journal = clean_text(journal_tag.get_text(' ', strip=True)) if journal_tag else ''

    doi_tag = item.select_one('.DOIResults')
    doi = clean_text(doi_tag.get_text(' ', strip=True)).replace('DOI:', '').strip() if doi_tag else ''

    abstract_parts = [
        clean_text(tag.get_text(' ', strip=True))
        for tag in item.select('.abstract')
        if clean_text(tag.get_text(' ', strip=True))
    ]

    return {
        'scielo_id': item.get('id', ''),
        'title': title,
        'link': link,
        'authors': authors,
        'journal': journal,
        'year': extract_year_from_source(source_text),
        'source_text': source_text,
        'doi': doi,
        'abstract': ' | '.join(abstract_parts),
    }


def parse_scielo_search_page(html):
    """Extrai todos os resultados e metadados gerais de uma página SciELO."""
    soup = BeautifulSoup(html, 'lxml')

    total_hits_tag = soup.select_one('#TotalHits')
    total_hits = clean_text(total_hits_tag.get_text()) if total_hits_tag else ''
    total_hits_int = int(re.sub(r'\D+', '', total_hits)) if re.sub(r'\D+', '', total_hits) else None

    page_tag = soup.select_one('input.goto_page')
    page_value = clean_text(page_tag.get('value')) if page_tag else ''

    items = soup.select('#ResultArea .results .item')

    rows = []
    for item in items:
        row = parse_scielo_result_item(item)
        if row['title'] or row['link'] or row['doi']:
            row['total_hits_on_search'] = total_hits_int
            row['page_value'] = page_value
            rows.append(row)

    return rows, total_hits_int

## 4. Scraper por keyword

In [ ]:
def scrape_scielo_keyword(
    keyword,
    driver,
    page_size=15,
    max_pages=None,
    collection='scl',
    language='pt',
    sleep_seconds=3,
):
    """Coleta todos os resultados de uma keyword na SciELO."""
    keyword_slug = slugify_keyword(keyword)
    rows = []
    total_hits = None

    page_index = 0
    pbar = tqdm(desc=f'keyword={keyword}', unit='page')

    while True:
        if max_pages is not None and page_index >= max_pages:
            break

        if total_hits is not None:
            expected_pages = math.ceil(total_hits / page_size)
            if page_index >= expected_pages:
                break

        url = build_scielo_search_url(
            keyword=keyword,
            page_index=page_index,
            page_size=page_size,
            collection=collection,
            language=language,
        )

        driver.get(url)
        time.sleep(sleep_seconds)
        html = driver.page_source

        if page_index == 0:
            debug_path = LOG_DIR / f'debug_search_{keyword_slug}_page_1.html'
            debug_path.write_text(html, encoding='utf-8')

        page_rows, page_total_hits = parse_scielo_search_page(html)

        if total_hits is None and page_total_hits is not None:
            total_hits = page_total_hits

        if not page_rows:
            print(f'Nenhum resultado extraído para {keyword} na página {page_index + 1}.')
            break

        for row in page_rows:
            row['search_keyword'] = keyword
            row['search_keyword_slug'] = keyword_slug
            row['search_page'] = page_index + 1
            row['search_offset'] = page_index * page_size
            row['search_url'] = url

        rows.extend(page_rows)
        page_index += 1
        pbar.update(1)

    pbar.close()

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.drop_duplicates(subset=['scielo_id', 'title', 'doi', 'link'])

    csv_path = SEARCH_DIR / f'scielo_{keyword_slug}_search_results.csv'
    json_path = SEARCH_DIR / f'scielo_{keyword_slug}_search_results.json'

    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    json_path.write_text(json.dumps(df.to_dict(orient='records'), ensure_ascii=False, indent=2), encoding='utf-8')

    print(f'{keyword}: {len(df)} resultados salvos em {csv_path}')
    return df

## 5. Rodar para todas as keywords

In [ ]:
driver = make_driver(headless=HEADLESS)

all_search_results = []

try:
    for keyword in KEYWORDS:
        keyword_df = scrape_scielo_keyword(
            keyword=keyword,
            driver=driver,
            page_size=PAGE_SIZE,
            max_pages=MAX_PAGES,
            collection=COLLECTION,
            language=LANGUAGE,
            sleep_seconds=SLEEP_SECONDS,
        )
        all_search_results.append(keyword_df)

finally:
    driver.quit()

combined_search = pd.concat(all_search_results, ignore_index=True) if all_search_results else pd.DataFrame()

combined_path = SEARCH_DIR / 'scielo_search_results_combined.csv'
combined_search.to_csv(combined_path, index=False, encoding='utf-8-sig')

print(combined_search.shape)
combined_search.head()

## 6. Diagnóstico da coleta

In [ ]:
if combined_search.empty:
    print('Nenhum resultado coletado.')
else:
    diagnostic_search = (
        combined_search
        .groupby('search_keyword', as_index=False)
        .agg(
            records=('title', 'count'),
            unique_titles=('title', 'nunique'),
            unique_dois=('doi', lambda x: x.replace('', pd.NA).dropna().nunique()),
            unique_journals=('journal', lambda x: x.replace('', pd.NA).dropna().nunique()),
            min_year=('year', lambda x: pd.to_numeric(x, errors='coerce').min()),
            max_year=('year', lambda x: pd.to_numeric(x, errors='coerce').max()),
        )
        .sort_values('records', ascending=False)
    )

    diagnostic_search.to_csv(SEARCH_DIR / 'diagnostic_search_by_keyword.csv', index=False, encoding='utf-8-sig')
    display(diagnostic_search)